# 🇻🇳 PhoBERT × UIT-VSMEC — Vietnamese Emotion Classification

| | |
|---|---|
| **Model** | `vinai/phobert-base-v2` |
| **Dataset** | `tridm/UIT-VSMEC` (7 classes, ~5.5k train) |
| **Platform** | Kaggle T4 (16 GB VRAM) |
| **Hub** | `DVQ290703/phobert-base-v2-uit-vsmec` |

**Flow:** Load → Encode → Tokenise → Train (weighted loss) → Eval → Push to Hub

In [3]:
!pip install -q transformers datasets evaluate scikit-learn accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.9 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
c

In [61]:
# ============================================================
# LOAD MODEL + LORA — Hỗ trợ resume từ HuggingFace Hub
# ============================================================

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# ── Login HuggingFace ──────────────────────────────────────
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✓ HuggingFace login OK")

✓ HuggingFace login OK


In [63]:
import os, torch

# ── toggle ──────────────────────────────────────────────────
SMOKE_RUN     = False    # False → full train + push to Hub
SMOKE_SAMPLES = 500     # số train samples khi smoke

# ── model / hub ─────────────────────────────────────────────
MODEL_NAME = "vinai/phobert-base-v2"
HUB_REPO   = "qdovan03/phobert-base-v2-uit-vsmec"
HF_TOKEN   = os.environ.get("HF_TOKEN")  

# ── paths / columns ─────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/phobert-vsmec"
TEXT_COL   = "Sentence"
LABEL_COL  = "Emotion"

# ── hyperparams ─────────────────────────────────────────────
MAX_LEN      = 128
BATCH_SIZE   = 32
EVAL_BATCH   = 64
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
NUM_EPOCHS   = 3 if SMOKE_RUN else 20
EARLY_STOP   = 3    
SEED         = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")
print(f"Config  : SMOKE_RUN={SMOKE_RUN} | epochs={NUM_EPOCHS} | batch={BATCH_SIZE}")

Device  : Tesla T4
Config  : SMOKE_RUN=False | epochs=20 | batch=32


In [34]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed,
)
import evaluate
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight

set_seed(SEED)

In [19]:
LABEL_NAMES = ["Anger", "Disgust", "Enjoyment", "Fear", "Other", "Sadness", "Surprise"]
label2id    = {l: i for i, l in enumerate(LABEL_NAMES)}
id2label    = {i: l for l, i in label2id.items()}
NUM_LABELS  = len(LABEL_NAMES)

print(f"Labels ({NUM_LABELS}):")
for name, idx in label2id.items():
    print(f"  {idx}  {name}")

Labels (7):
  0  Anger
  1  Disgust
  2  Enjoyment
  3  Fear
  4  Other
  5  Sadness
  6  Surprise


In [20]:
raw = load_dataset("tridm/UIT-VSMEC")
print(raw)

# ── sanity check labels ──────────────────────────────────────
unexpected = set(raw["train"].unique(LABEL_COL)) - set(LABEL_NAMES)
assert not unexpected, f"Unexpected labels in dataset: {unexpected}"

# ── lưu label IDs của FULL train để tính class weights đúng ──
full_train_label_ids = [label2id[l] for l in raw["train"][LABEL_COL]]

# ── subsample nếu SMOKE ──────────────────────────────────────
if SMOKE_RUN:
    raw["train"]      = raw["train"].shuffle(seed=SEED).select(range(SMOKE_SAMPLES))
    raw["validation"] = raw["validation"].shuffle(seed=SEED).select(range(100))
    raw["test"]       = raw["test"].shuffle(seed=SEED).select(range(100))

print(f"\nSizes → train: {len(raw['train'])}  val: {len(raw['validation'])}  test: {len(raw['test'])}")

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 5548
    })
    validation: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 686
    })
    test: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 693
    })
})

Sizes → train: 5548  val: 686  test: 693


In [21]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_LABELS),
    y=full_train_label_ids,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("Class weights (balanced):")
for name, w in zip(LABEL_NAMES, class_weights):
    bar = "█" * int(w * 8)
    print(f"  {name:<12} {w:.4f}  {bar}")

Class weights (balanced):
  Anger        2.0270  ████████████████
  Disgust      0.7400  █████
  Enjoyment    0.5087  ████
  Fear         2.4924  ███████████████████
  Other        0.7763  ██████
  Sadness      0.8369  ██████
  Surprise     3.2751  ██████████████████████████


In [22]:
# ── encode string labels → integer ids ──────────────────────
def encode_labels(batch):
    batch["labels"] = [label2id[l] for l in batch[LABEL_COL]]
    return batch

raw = raw.map(encode_labels, batched=True)

# ── tokenise ────────────────────────────────────────────────
# NOTE: PhoBERT được train trên text đã word-segment bằng VnCoreNLP.
# Với text mạng xã hội ngắn (UIT-VSMEC), dùng BPE thô vẫn cho kết quả
# tốt và giữ pipeline đơn giản trên Kaggle.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,      # dynamic padding → DataCollatorWithPadding
    )

tokenized = raw.map(
    tokenize,
    batched=True,
    remove_columns=[TEXT_COL, LABEL_COL],
)
tokenized.set_format("torch")
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ── preview một sample ──────────────────────────────────────
sample = tokenized["train"][0]
print("Sample keys :", list(sample.keys()))
print("input_ids   :", sample["input_ids"][:10], "...")
print("label       :", sample["labels"].item(), "→", id2label[sample["labels"].item()])

Map:   0%|          | 0/5548 [00:00<?, ? examples/s]

Map:   0%|          | 0/686 [00:00<?, ? examples/s]

Map:   0%|          | 0/693 [00:00<?, ? examples/s]

Map:   0%|          | 0/5548 [00:00<?, ? examples/s]

Map:   0%|          | 0/686 [00:00<?, ? examples/s]

Map:   0%|          | 0/693 [00:00<?, ? examples/s]

Sample keys : ['labels', 'input_ids', 'attention_mask']
input_ids   : tensor([  0,  13,  68, 611, 387, 737, 221,   8, 148,  15]) ...
label       : 4 → Other


In [23]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

total_params  = sum(p.numel() for p in model.parameters())
train_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params / 1e6:.1f} M")
print(f"Trainable params : {train_params / 1e6:.1f} M")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params     : 135.0 M
Trainable params : 135.0 M


In [24]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights  # stored on CPU, moved per step

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # **kwargs hứng extra args của Transformers mới (e.g. num_items_in_batch)
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        weights = (
            self._class_weights.to(logits.device)
            if self._class_weights is not None
            else None
        )
        loss = torch.nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [25]:
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1_weighted": f1}

In [64]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── schedule ────────────────────────────────────────────
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    # ── precision ───────────────────────────────────────────
    fp16=True,

    # ── eval / checkpoint ───────────────────────────────────
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,         # giữ tối đa 2 checkpoint → tiết kiệm disk

    # ── logging ─────────────────────────────────────────────
    logging_steps=20,
    report_to="none",

    # ── seed ────────────────────────────────────────────────
    seed=SEED,
    data_seed=SEED,

    # ── hub (dùng khi push_to_hub thủ công cuối notebook) ──
    hub_model_id=HUB_REPO,
    hub_token=HF_TOKEN,
    push_to_hub=False,
)

print("Training args OK")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training args OK


In [27]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,   # Transformers ≥ 4.47
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP)],
    class_weights=class_weights_tensor,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.912938,1.869094,0.269679,0.242416
2,1.620677,1.448007,0.517493,0.517229
3,1.284827,1.232165,0.549563,0.552209
4,1.011956,1.134041,0.596210,0.596186
5,0.903334,1.133314,0.580175,0.586475
6,0.721892,1.130100,0.581633,0.587268
7,0.623139,1.167375,0.612245,0.618616
8,0.520177,1.227154,0.628280,0.632655
9,0.394077,1.211991,0.615160,0.621469
10,0.361433,1.308836,0.618076,0.621666


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=957, training_loss=0.8924642296794066, metrics={'train_runtime': 482.3403, 'train_samples_per_second': 230.045, 'train_steps_per_second': 3.607, 'total_flos': 1914796495799400.0, 'train_loss': 0.8924642296794066, 'epoch': 11.0})

In [28]:
test_out = trainer.predict(tokenized["test"])
pred_ids = np.argmax(test_out.predictions, axis=-1)
true_ids = test_out.label_ids

wf1 = f1_score(true_ids, pred_ids, average="weighted")
print(f"Weighted F1 (test): {wf1:.4f}\n")
print(classification_report(true_ids, pred_ids, target_names=LABEL_NAMES, digits=4))

Weighted F1 (test): 0.6422

              precision    recall  f1-score   support

       Anger     0.5938    0.4750    0.5278        40
     Disgust     0.6900    0.5227    0.5948       132
   Enjoyment     0.7665    0.6632    0.7111       193
        Fear     0.5781    0.8043    0.6727        46
       Other     0.5357    0.5814    0.5576       129
     Sadness     0.6164    0.7759    0.6870       116
    Surprise     0.6364    0.7568    0.6914        37

    accuracy                         0.6436       693
   macro avg     0.6310    0.6542    0.6346       693
weighted avg     0.6544    0.6436    0.6422       693



## 13 · Push to Hub 🤗
Chỉ chạy khi `SMOKE_RUN = False`.

In [67]:
from huggingface_hub import login
login(token=HF_TOKEN)

# Sau đó push bình thường
trainer.push_to_hub(commit_message="PhoBERT-base-v2 fine-tuned on UIT-VSMEC")
tokenizer.push_to_hub(HUB_REPO, token=HF_TOKEN)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/qdovan03/phobert-base-v2-uit-vsmec/commit/533fa7589d79f05ac0c3a924d07b7e46fa49a976', commit_message='Upload tokenizer', commit_description='', oid='533fa7589d79f05ac0c3a924d07b7e46fa49a976', pr_url=None, repo_url=RepoUrl('https://huggingface.co/qdovan03/phobert-base-v2-uit-vsmec', endpoint='https://huggingface.co', repo_type='model', repo_id='qdovan03/phobert-base-v2-uit-vsmec'), pr_revision=None, pr_num=None)